# Bonus: building DNA systems for MD

Thor van Heesch (T.vanHeesch@uva.nl) and Jocelyne Vreede (J.Vreede@uva.nl)

van 't Hoff Institute for Molecular Sciences, University of Amsterdam

===

This notebook is for participants who finished `1_analyze.ipynb` and `2_build.ipynb`. It shows more of what mdna can build for MD simulations: DNA bound to proteins, DNA loops, chemical modifications, supercoiled and interlocked DNA.

The sections are independent, so pick the ones you like:

1. **A nucleosome with linker DNA.** Extend the DNA of a crystal structure, and make several starting structures.
2. **Connecting DNA.** Fill the gap between two pieces of DNA, and fold DNA into a loop between proteins.
3. **Modifications your force field may not know.** Methylation, Hoogsteen base pairs and non-canonical bases.
4. **Supercoiled minicircles.** Where does extra linking number go, and an atomistic movie of supercoiling. Takes about 5 minutes to run.
5. **Interlocked DNA rings.** Build a DNA catenane and compute the linking number between two rings.
6. **Before you simulate.** Export for AMBER force fields, overlaps and the size of the water box.

Sections 1 and 2 download a structure from the Protein Data Bank, so they need an internet connection. Each section starts with an example and ends with questions. Run the import cell below first.

In [ ]:
import os
import urllib.request

import numpy as np
import mdtraj as md
import matplotlib.pyplot as plt

import devtools.mdna_workarounds  # fixes for known issues in mdna 0.2.0, see devtools/mdna_workarounds.py
import mdna
from devtools.viewer import view  # 3D viewer inside the notebook, see devtools/viewer.py

os.makedirs('data', exist_ok=True)

## 1. A nucleosome with linker DNA

In the cell, DNA is wrapped around histone proteins. A nucleosome core particle contains 147 bp of DNA wrapped around eight histones. Here you analyse the DNA in the crystal structure 1KX5 (Davey et al., J. Mol. Biol. 2002) with the tools from `1_analyze.ipynb`. Then you add linker DNA on both sides, as in chromatin.

In [ ]:
# download the structure from the Protein Data Bank (only the first time)
if not os.path.exists('data/1kx5.pdb'):
    urllib.request.urlretrieve('https://files.rcsb.org/download/1KX5.pdb', 'data/1kx5.pdb')

pdb = md.load('data/1kx5.pdb')
print(pdb)
for chain in pdb.top.chains:
    print(f'chain {chain.index}: {chain.n_residues} residues, first residue {chain.residue(0).name}')

Chains 0 and 1 are the two DNA strands and chains 2 to 9 are the histones. The other chains are ions and water. Load the DNA into mdna:

In [ ]:
nucleosome = mdna.load(pdb, chainids=[0, 1])
nucleosome.describe()
view(nucleosome)

Plot the roll along the DNA:

In [ ]:
params, names = nucleosome.get_parameters(step=True)
roll = params[0, :, names.index('roll')]

fig, ax = plt.subplots(figsize=[8, 3])
ax.plot(roll, '-o', ms=3)
ax.axhline(0, color='grey', lw=0.5)
ax.set_xlabel('Base pair step')
ax.set_ylabel('roll [degrees]')

Now add 30 bp of linker DNA to each end with `extend()`. The new DNA starts straight and is then minimized with MC. Excluded volume is switched off (`exvol_rad=0`): the wrapped DNA comes close to itself, so the excluded-volume beads would overlap from the start.

In [ ]:
nucleosome.extend(n_bp=30, exvol_rad=0)                 # at the 3' end
nucleosome.extend(n_bp=30, exvol_rad=0, forward=False)  # at the 5' end
nucleosome.describe()
view(nucleosome)

Finally, put the histones back and save the complex:

`view(complex_traj)` shows the complex in 3D, inside the notebook. Drag with the left mouse button to rotate, with the right mouse button to pan, and scroll to zoom. The viewer loads Mol* from the internet, so you need a connection.

In [ ]:
histones = pdb.atom_slice(pdb.top.select('protein'))
complex_traj = nucleosome.get_traj().stack(histones)
print(complex_traj)
complex_traj.save_pdb('./nucleosome_with_linkers.pdb')
view(complex_traj)

### Questions

a. The roll goes up and down along the DNA. Estimate the period in base pairs. How is this related to the way the DNA bends around the histones? (Hint: a positive roll bends the DNA towards its major groove.)

b. Compare the mean twist of the nucleosomal DNA with the 34.3 degrees of ideal B-DNA. How many base pairs per turn is that?

c. The linkers have a random sequence. Load the nucleosome again and extend it with a sequence you choose, for example `sequence='A'*30`.

d. Open `nucleosome_with_linkers.pdb` in a viewer. Do the new linkers run into the histones? Check it in code with `md.compute_neighbors()`.

e. MD simulations are often repeated from several different starting structures (replicas). The MC minimization in `extend()` runs at 300 K, so each run gives a different linker conformation. Build three nucleosomes with linkers and save them as separate PDB files. How different are they?

In [ ]:
# your code here


## 2. Connecting DNA

`mdna.connect()` joins the 3' end of one DNA to the 5' end of another DNA, and fills the gap between them with new DNA. It chooses the number of new base pairs so that they fit the gap and their twist matches both ends. Then it relaxes the new DNA with a short MC minimization, while the two original pieces stay in place.

To test it, cut 12 bp out of the middle of a 36 bp DNA and let `connect()` fill the gap. mdna describes a DNA by the reference frames of its base pairs (`get_frames()` in `1_analyze.ipynb`), so you can cut the DNA by taking slices of its frames:

In [ ]:
dna = mdna.make(n_bp=36)
frames = dna.get_frames()[:, -1]   # (n_bp, 4, 3): the origin and three axes of each base pair
left = mdna.load(frames=frames[:12], sequence=dna.sequence[:12])
right = mdna.load(frames=frames[24:], sequence=dna.sequence[24:])

joined = mdna.connect(left, right)
joined.describe()

fig = plt.figure(figsize=[10, 4])
ax = fig.add_subplot(121, projection='3d')
left.draw(ax=ax, color_lead='tab:blue', color_anti='tab:blue')
right.draw(ax=ax, color_lead='tab:red', color_anti='tab:red')
ax.set_title('two pieces')
ax = fig.add_subplot(122, projection='3d')
joined.draw(ax=ax)
ax.set_title('after connect()')

view(joined)

The function below prints the mean rise and twist of the new DNA, including the two steps that join it to the original pieces:

In [ ]:
def linker_shape(joined, left, right):
    """Mean rise and twist of the base-pair steps that involve the new DNA."""
    params, names = joined.get_parameters(step=True)
    new = params[-1, left.n_bp:joined.n_bp - right.n_bp + 1]
    n_new = joined.n_bp - left.n_bp - right.n_bp
    print(f'{n_new} new base pairs: rise {new[:, names.index("rise")].mean():.2f} Å, twist {new[:, names.index("twist")].mean():.1f} degrees')

linker_shape(joined, left, right)

### Questions

a. How many base pairs does `connect()` put in the gap? Compare with the 12 bp you cut out, and compare the rise and twist of the new DNA with those of B-DNA (3.4 Å and 34.3 degrees).

b. Move the right piece 1 nm further away and connect again. The DNA was built along the x axis, so add 1 nm to the x coordinate of its origins: `moved = frames[24:].copy()`, `moved[:, 0] += [1, 0, 0]`, and load it with `mdna.load(frames=moved, sequence=dna.sequence[24:])`. How many base pairs does `connect()` choose now, and what is the rise? Is this a good model? (Hint: the new DNA has to match the twist at both ends, not only the length of the gap.)

c. Choose the number of base pairs yourself, for example `mdna.connect(left, moved_right, n_bp=15)`, and try a few values. How does the new DNA fill the gap now? With `sequence=...` instead of `n_bp` you also choose the sequence of the new DNA.

In [ ]:
# your code here

### A DNA loop between proteins

Proteins that bind DNA at two places can fold the DNA in between into a loop. Here you use the structure 8SRP, in which FOXP3 proteins form a ladder between two DNA duplexes, and connect the two duplexes into one DNA loop.

In [ ]:
# download the structure from the Protein Data Bank (only the first time)
if not os.path.exists('data/8srp.pdb'):
    urllib.request.urlretrieve('https://files.rcsb.org/download/8SRP.pdb', 'data/8srp.pdb')

foxp3 = md.load('data/8srp.pdb')   # mdtraj may warn about the unit cell; you can ignore that
for chain in foxp3.top.chains:
    print(f'chain {chain.index}: {chain.n_residues} residues, first residue {chain.residue(0).name}')

Chains 2 and 3 form one DNA duplex, chains 4 and 5 the other. Load both, and find the two ends that should be connected: the last base pair of duplex a and the first base pair of duplex b.

In [ ]:
dna_a = mdna.load(foxp3, chainids=[2, 3])
dna_b = mdna.load(foxp3, chainids=[4, 5])
protein = foxp3.atom_slice(foxp3.top.select('protein'))

start = dna_a.get_frames()[-1, 0, 0]   # last base pair of duplex a
end = dna_b.get_frames()[0, 0, 0]      # first base pair of duplex b
print(f'distance between the two ends: {np.linalg.norm(end - start):.1f} nm')

To make a loop instead of a short straight connection, give `connect()` control points that lead the DNA away from the protein: in the direction from the centre of the protein to the middle of the two ends.

In [ ]:
middle = (start + end) / 2
away = middle - protein.xyz[0].mean(axis=0)
away /= np.linalg.norm(away)

height = 15  # how far the loop sticks out, in nm
control_points = np.array([start, start + 3 * away, middle + height * away, end + 3 * away, end])

loop = mdna.connect(dna_a, dna_b, control_points=control_points)
loop.describe()

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
loop.draw(ax=ax)
ax.plot(*control_points.T, 'o-', color='tab:red')

In [ ]:
complex_loop = protein.stack(loop.get_traj())
complex_loop.save_pdb('./foxp3_dna_loop.pdb')
print(complex_loop)
view(complex_loop)

### Questions

a. How many base pairs are in the loop? Change `height` and see how the size of the loop changes.

b. Connect the two duplexes without control points: `mdna.connect(dna_a, dna_b)`. How many base pairs does the straight connection get, and what are its rise and twist (use `linker_shape()` from above)? Why is a loop an easier way to connect these two duplexes?

c. Count the protein atoms within 0.2 nm of the DNA in `complex_loop`, and find out which nucleotides they touch. Compare with the crystal structure `foxp3`. Where do the close contacts come from?

In [ ]:
# your code here


## 3. Modifications your force field may not know

mdna can add chemical modifications and non-canonical bases (see the last slides of the lecture). Building them is easy. Simulating them requires force-field parameters for every new residue, so always check which residue names end up in your PDB file.

In [ ]:
dna = mdna.make(sequence='GCGCAATTGCGC')
dna.methylate(CpG=True)   # 5-methylcytosine at every CpG site
print([residue.name for residue in dna.get_traj().top.residues])

In a Hoogsteen base pair, the purine is rotated by 180 degrees around its glycosidic bond, so it pairs with a different edge. Flip base pair 5 (an A-T pair) and compare the base-pair parameters with the normal Watson-Crick structure:

In [ ]:
watson_crick = mdna.make(sequence='GCGCAATTGCGC')
hoogsteen = mdna.make(sequence='GCGCAATTGCGC')
hoogsteen.flip(fliplist=[5], deg=180)
hoogsteen.save_pdb('./hoogsteen')
view(hoogsteen)

for label, obj in [('Watson-Crick', watson_crick), ('Hoogsteen', hoogsteen)]:
    params, names = obj.get_parameters(base=True)
    print(label, {name: round(float(params[0, 5, i]), 1) for i, name in enumerate(names)})

### Questions

a. Which residue name does mdna use for 5-methylcytosine? Would a standard DNA force field recognise it?

b. Which base-pair parameters change when you flip the base, and why?

c. In a real Hoogsteen base pair the C1' atoms of the two nucleotides are about 0.2 nm closer together than in a Watson-Crick pair. Measure the C1'-C1' distance of base pair 5 in both structures. Does mdna change it?

d. Build a duplex with the fluorescent base 2-aminopurine (`E`), and one with the Hachimoji bases (`B`, `S`, `P`, `Z`). Which residue names do you get?

In [ ]:
# your code here


## 4. Supercoiled minicircles

In `2_build.ipynb` you saw that minimizing an overwound minicircle turns part of the extra twist into writhe, while Lk = Tw + Wr stays the same. Here you measure how this depends on the amount of overwinding. Each minimization takes about a minute, so the next cell runs for about 5 minutes.

In [ ]:
dLk_values = [0, 4, 8, 12]
results = []
for dLk in dLk_values:
    circle = mdna.make(n_bp=200, circular=True, dLk=dLk)
    circle.minimize(equilibrate_writhe=True)
    circle.save_pdb(f'./minicircle_200bp_dLk{dLk}')
    results.append(circle.get_linking_number())   # [Lk, Wr, Tw]
results = np.array(results)

In [ ]:
fig, ax = plt.subplots(figsize=[4, 3])
ax.plot(dLk_values, results[:, 1], 'o-', label='writhe Wr')
ax.plot(dLk_values, results[:, 2] - results[0, 2], 'o-', label='extra twist Tw - Tw(dLk=0)')
ax.set_xlabel('dLk')
ax.set_ylabel('turns')
ax.legend()

In [ ]:
# The saved minicircles, from relaxed (dLk = 0) to strongly overwound (dLk = 12)
for dLk in dLk_values:
    view(md.load(f'./minicircle_200bp_dLk{dLk}.pdb'))

### From the coarse-grained model back to atoms

`minimize()` runs a Monte Carlo simulation of a coarse-grained model, in which every base pair is a rigid body. It keeps snapshots of this simulation, and mdna can build an atomistic structure for each of them. This is called backmapping. Use it to make an atomistic movie of how the last minicircle (dLk = 12) supercoils. The same approach gives you several different starting structures for MD replicas.

The snapshots are stored in `circle.minimizer.out`, an internal part of mdna that may change in future versions.

In [ ]:
confs = circle.minimizer.out['confs']   # (n_snapshots, n_bp, 4, 4): position and orientation of each base pair
print(confs.shape)

def snapshot_frames(conf):
    """Convert one MC snapshot to mdna reference frames: an origin and three axes per base pair."""
    frames = np.empty((conf.shape[0], 4, 3))
    frames[:, 0] = conf[:, :3, 3]
    frames[:, 1:] = conf[:, :3, :3].transpose(0, 2, 1)
    return frames

snapshots = []
for k in np.linspace(0, len(confs) - 1, 10).astype(int):
    snapshot = mdna.load(frames=snapshot_frames(confs[k]), sequence=circle.sequence, circular=True)
    snapshots.append(snapshot.get_traj())
    print(f'snapshot {k}: [Lk, Wr, Tw] =', np.round(snapshot.get_linking_number(), 2))

movie = md.join(snapshots)
movie[0].save_pdb('./supercoiling_movie.pdb')
movie.save_xtc('./supercoiling_movie.xtc')
print(movie)
view(movie)

### Questions

a. What fraction of the extra linking number ends up as writhe? Is that fraction the same for each dLk?

b. Look at the saved minicircles in a viewer. From which dLk does the minicircle clearly bend out of its plane?

c. Supercoiling is often given as the superhelical density σ = dLk / Lk0, with Lk0 = n_bp / 10.5 the linking number of relaxed DNA. Bacterial plasmids have σ ≈ -0.06. Which dLk is that for a 200 bp minicircle? Compare it with the values used here.

d. Repeat the experiment with negative dLk (underwound DNA, as in most cells). Does the DNA respond in the same way?

e. Play the movie above (or load `supercoiling_movie.pdb` together with `supercoiling_movie.xtc` in a viewer). How does the shape change, and how does that match the printed writhe?

In [ ]:
# your code here


## 5. Interlocked DNA rings

Some organisms keep DNA as interlocked rings. In trypanosomes, the parasites that cause sleeping sickness, the mitochondrial DNA (kinetoplast DNA) is a network of thousands of small DNA circles, linked together like chain mail. Two linked rings cannot be separated without cutting one of them, even though no bond connects them.

How strongly two closed curves are linked is measured by their linking number, computed with Gauss's linking integral. This is the same quantity as the linking number Lk between the two strands of a minicircle.

In [ ]:
def ring(radius, center, plane):
    """Control points of a circle in the 'xy', 'xz' or 'yz' plane. The first and last point are the same."""
    t = np.linspace(0, 2 * np.pi, 60)
    a, b, zero = radius * np.cos(t), radius * np.sin(t), np.zeros_like(t)
    points = {'xy': [a, b, zero], 'xz': [a, zero, b], 'yz': [zero, a, b]}[plane]
    return np.array(points).T + np.array(center)

def linking_number(curve1, curve2):
    """Gauss linking number of two closed curves, each an (n, 3) array whose first and last point are the same."""
    d1, d2 = np.diff(curve1, axis=0), np.diff(curve2, axis=0)   # short segments along each curve
    m1, m2 = curve1[:-1] + d1 / 2, curve2[:-1] + d2 / 2         # midpoints of the segments
    r = m1[:, None, :] - m2[None, :, :]
    cross = np.cross(d1[:, None, :], d2[None, :, :])
    integrand = np.einsum('ijk,ijk->ij', r, cross) / np.linalg.norm(r, axis=-1)**3
    return integrand.sum() / (4 * np.pi)

In [ ]:
radius = 10  # nm
points1 = ring(radius, center=[0, 0, 0], plane='xy')
points2 = ring(radius, center=[radius, 0, 0], plane='xz')   # passes through the hole of ring 1
print('linking number of the two curves:', round(linking_number(points1, points2), 2))

ring1 = mdna.make(control_points=points1, circular=True)
ring2 = mdna.make(control_points=points2, circular=True)

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ring1.draw(ax=ax, color_lead='tab:blue', color_anti='tab:blue')
ring2.draw(ax=ax, color_lead='tab:red', color_anti='tab:red')

The linking number of the DNA itself follows from the helix axes, the curves through the base-pair centres. Save both rings in one PDB file to look at them in a viewer:

In [ ]:
def axis(dna):
    """Closed curve through the base-pair centres of a circular DNA."""
    origins = dna.get_frames()[:, -1, 0]
    return np.vstack([origins, origins[:1]])

print('linking number of the two DNA rings:', round(linking_number(axis(ring1), axis(ring2)), 2))

catenane = ring1.get_traj().stack(ring2.get_traj())
catenane.save_pdb('./catenane.pdb')
print(catenane)
view(catenane)

### Questions

a. Move ring 2 so that it no longer passes through ring 1, for example `center=[2.5 * radius, 0, 0]`. What is the linking number now?

b. What happens to the sign of the linking number when you reverse the direction of one ring (`points2[::-1]`)?

c. Build a chain of five rings, each linked to the next, and save it as one PDB file. Compute the linking number of every pair of rings: which pairs are linked? (Hint: place the centres 1.5 radius apart along x and alternate between the 'xy' and 'xz' planes.)

d. Thinking question: the enzyme topoisomerase II separates linked rings. It cuts both strands of one ring, passes the other ring through the gap and seals the cut. By how much does the linking number between the two rings change in one such step?

In [ ]:
# your code here


## 6. Before you simulate

The energy minimization of your MD engine repairs small errors in a model, for example bonds that are slightly too long where mdna joins its nucleotide templates. Some things it cannot fix, and some you have to sort out before the MD engine even accepts the structure.

### Terminal phosphates

`save_pdb()` writes a phosphate group at the 5' end of each strand. The 5' terminal nucleotides of AMBER force fields have no phosphate, so for example `gmx pdb2gmx` stops with an error. `mdna.sequence_to_pdb(sequence='CGCGAATTCGCG', filename='my_dna', output='GROMACS')` writes a linear DNA without them. The function below does the same for any linear mdna structure:

In [ ]:
def save_without_5prime_phosphates(dna, filename):
    """Save a linear DNA without the phosphate group at the 5' end of each strand."""
    traj = dna.get_traj()
    first_residues = [chain.residue(0) for chain in traj.top.chains]   # each strand starts at its 5' end
    remove = {atom.index for residue in first_residues for atom in residue.atoms if atom.name in ('P', 'OP1', 'OP2')}
    keep = [atom.index for atom in traj.top.atoms if atom.index not in remove]
    traj.atom_slice(keep).save_pdb(filename)

dna = mdna.make(sequence='CGCGAATTCGCG')
save_without_5prime_phosphates(dna, './drew_dickerson_amber.pdb')
print('first atoms of strand 1:', [atom.name for atom in md.load('./drew_dickerson_amber.pdb').top.residue(0).atoms][:4])

### Overlapping DNA

In a knot, distant parts of the DNA come close to each other. DNA is about 2 nm wide, so the helix axes of two parts of the molecule should stay more than 2 nm apart. An energy minimization cannot pull DNA strands that pass through each other apart again. The function below finds the closest approach between base pairs that are at least 20 base pairs apart along the chain:

In [ ]:
def closest_approach(dna, min_separation=20):
    """Smallest distance (nm) between base-pair centres that are at least min_separation base pairs apart."""
    origins = dna.get_frames()[:, -1, 0]
    n = len(origins)
    distances = np.linalg.norm(origins[:, None] - origins[None, :], axis=-1)
    i, j = np.indices(distances.shape)
    separation = np.abs(i - j)
    if dna.circular:
        separation = np.minimum(separation, n - separation)
    return distances[separation >= min_separation].min()

def parametric_trefoil(t, scaling=5):
    x = scaling*(np.sin(t)) + scaling*(2*np.sin(2*t))
    y = scaling*(np.cos(t)) - scaling*(2*np.cos(2*t))
    z = scaling*(-np.sin(3*t))
    return x, y, z

t = np.linspace(0, 2 * np.pi, 50)
knot = mdna.make(control_points=np.array(parametric_trefoil(t, scaling=5)).T, circular=True)
print(f'{knot.n_bp} bp, closest approach {closest_approach(knot):.2f} nm')

### How big will the system be?

In an MD simulation the DNA sits in a box of water, usually with at least 1 nm of water between the DNA and the edge of the box. The function below estimates the size of a cubic box, the number of water molecules (about 33 per nm³) and the number of sodium ions that make the system neutral (one per phosphate).

In [ ]:
def system_size(dna, padding=1.0):
    """Rough size of the DNA in a cubic water box with `padding` nm of water on every side."""
    traj = dna.get_traj()
    xyz = traj.xyz[-1]
    box = (xyz.max(axis=0) - xyz.min(axis=0)).max() + 2 * padding  # edge of the box in nm
    waters = int(33.4 * box**3)
    ions = len(traj.top.select('name P'))
    dna_atoms = int(1.6 * traj.n_atoms)  # mdna writes no hydrogens; adding them gives about 60% more atoms
    total = dna_atoms + 3 * waters + ions
    print(f'{dna.n_bp} bp: box {box:.1f} nm, {waters:,} waters, {ions} Na+, about {total:,} atoms')

system_size(mdna.make(sequence='CGCGAATTCGCG'))

### Questions

a. Why should you not use `save_without_5prime_phosphates()` for a circular DNA?

b. Make the trefoil smaller by lowering `scaling`. Below which value do parts of the knot overlap?

c. Compare the system size of a straight 200 bp DNA (`mdna.make(n_bp=200)`) with a 200 bp minicircle (`mdna.make(n_bp=200, circular=True)`). Why do the numbers differ so much? How could you simulate a long linear DNA anyway?

In [ ]:
# your code here


## More to explore

The [mdna documentation](https://heezch.github.io/mdna/) has more tutorials, for example building DNA along a protein filament.